In [2]:
#r "task17/bin/Debug/net10.0/task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class TestCommand : ICommand
{
    private readonly int _id;
    private int _counter;

    public TestCommand(int id)
    {
        _id = id;
    }

    public void Execute()
    {
        Console.WriteLine($"Поток {_id} вызов {++_counter}");
    }
}

public class RepeatCommand : ILongCommand
{
    private readonly ICommand _inner;
    private readonly int _maxCalls;
    private int _calls;

    public bool IsCompleted => _calls >= _maxCalls;

    public RepeatCommand(ICommand inner, int maxCalls)
    {
        _inner = inner;
        _maxCalls = maxCalls;
    }

    public void Execute()
    {
        if (!IsCompleted)
        {
            _inner.Execute();
            _calls++;
        }
    }
}

Console.WriteLine("=== Демонстрация длительных операций ===\n");
Console.WriteLine("5 команд TestCommand, каждая выполняется 3 раза:\n");

var planner = new RoundRobinScheduler();
var worker = new ServerThread(planner);
worker.Start();

for (int i = 1; i <= 5; i++)
{
    planner.Add(new RepeatCommand(new TestCommand(i), 3));
}

while (planner.HasCommand())
{
    Thread.Sleep(50);
}

Thread.Sleep(100);

planner.Add(new HardStopCommand(worker));
worker.Join();

Console.WriteLine("\nПоток остановлен HardStop.\n");

Console.WriteLine("=== Замеры производительности ===\n");

var commandCounts = new int[] { 1, 2, 4, 8, 16, 32 };
int workload = 100000;
int stepsPerCommand = 10;
int warmupRuns = 2;
int measureRuns = 5;

var sequentialTimes = new double[commandCounts.Length];
var roundRobinTimes = new double[commandCounts.Length];

for (int i = 0; i < commandCounts.Length; i++)
{
    int n = commandCounts[i];

    double seqTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(workload, stepsPerCommand);
            while (!cmd.IsCompleted)
                cmd.Execute();
        }
        sw.Stop();
        if (run >= warmupRuns)
            seqTotal += sw.Elapsed.TotalMilliseconds;
    }
    sequentialTimes[i] = seqTotal / measureRuns;

    double rrTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var s = new RoundRobinScheduler();
        var st = new ServerThread(s);
        var commands = new HeavyCommand[n];
        
        for (int j = 0; j < n; j++)
        {
            commands[j] = new HeavyCommand(workload, stepsPerCommand);
            s.Add(commands[j]);
        }

        var sw = Stopwatch.StartNew();
        st.Start();

        while (!commands.All(c => c.IsCompleted))
            Thread.Sleep(1);

        sw.Stop();
        s.Add(new HardStopCommand(st));
        st.Join();

        if (run >= warmupRuns)
            rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    roundRobinTimes[i] = rrTotal / measureRuns;

    double diff = roundRobinTimes[i] - sequentialTimes[i];
    double diffPercent = (diff / sequentialTimes[i]) * 100.0;
    Console.WriteLine($"Команд: {n,3} | Послед: {sequentialTimes[i],8:F2} мс | Round Robin: {roundRobinTimes[i],8:F2} мс | Разница: {diffPercent,6:F1}%");
}

var plot1 = new Plot();
plot1.Title("Время выполнения: Последовательно vs Round Robin");
plot1.XLabel("Количество команд");
plot1.YLabel("Время (мс)");

var xs = commandCounts.Select(c => (double)c).ToArray();

var s1 = plot1.Add.Scatter(xs, sequentialTimes);
s1.LegendText = "Последовательно";
s1.MarkerSize = 8;

var s2 = plot1.Add.Scatter(xs, roundRobinTimes);
s2.LegendText = "Round Robin";
s2.MarkerSize = 8;

plot1.ShowLegend();
var fn1 = "plot_comparison.png";
plot1.SavePng(fn1, 800, 600);
display(HTML($"<img src='{fn1}?t={DateTime.Now.Ticks}' width='700'/>"));

var plot2 = new Plot();
plot2.Title("Накладные расходы Round Robin");
plot2.XLabel("Количество команд");
plot2.YLabel("Замедление (%)");

var overheads = new double[commandCounts.Length];
for (int i = 0; i < commandCounts.Length; i++)
    overheads[i] = ((roundRobinTimes[i] - sequentialTimes[i]) / sequentialTimes[i]) * 100.0;

var bars = plot2.Add.Bars(xs, overheads);
bars.LegendText = "Замедление";

var hLine = plot2.Add.HorizontalLine(15.0);
hLine.LineColor = ScottPlot.Color.FromHex("#FF0000");
hLine.LineWidth = 2;
hLine.LegendText = "Порог 15%";

plot2.ShowLegend();
var fn2 = "plot_overhead.png";
plot2.SavePng(fn2, 800, 600);
display(HTML($"<img src='{fn2}?t={DateTime.Now.Ticks}' width='700'/>"));

var reportPath = "report_task19.txt";
var sb = new System.Text.StringBuilder();

sb.AppendLine($"Отчёт по задаче 19: Длительные операции");
sb.AppendLine($"Дата: {DateTime.Now}");
sb.AppendLine();
sb.AppendLine("1. TestCommand реализует ICommand как в задании");
sb.AppendLine("2. RepeatCommand оборачивает TestCommand в ILongCommand для 3 вызовов");
sb.AppendLine("3. 5 экземпляров TestCommand выполнены по 3 раза через RoundRobinScheduler");
sb.AppendLine("4. Поток остановлен командой HardStop");
sb.AppendLine("5. Производительность Round Robin сопоставима с последовательным выполнением");

File.WriteAllText(reportPath, sb.ToString());
Console.WriteLine($"\nОтчёт сохранён: {reportPath}");

public class HeavyCommand : ILongCommand
{
    private readonly int _workload;
    private int _remaining;
    public bool IsCompleted => _remaining <= 0;

    public HeavyCommand(int workload, int steps)
    {
        _workload = workload;
        _remaining = steps;
    }

    public void Execute()
    {
        if (!IsCompleted)
        {
            _remaining--;
            double x = 0;
            for (int i = 0; i < _workload; i++)
                x += Math.Sin(i) * Math.Cos(i);
        }
    }
}

Installed Packages ScottPlot, 5.0.54

=== Демонстрация длительных операций ===

5 команд TestCommand, каждая выполняется 3 раза:

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 3 вызов 3
Поток 5 вызов 3
Поток 4 вызов 3
Поток 2 вызов 3

Поток остановлен HardStop.

=== Замеры производительности ===

Команд:   1 | Послед:    27.18 мс | Round Robin:    28.54 мс | Разница:    5.0%
Команд:   2 | Послед:    47.06 мс | Round Robin:    53.61 мс | Разница:   13.9%
Команд:   4 | Послед:    91.27 мс | Round Robin:   105.61 мс | Разница:   15.7%
Команд:   8 | Послед:   186.66 мс | Round Robin:   191.69 мс | Разница:    2.7%
Команд:  16 | Послед:   377.83 мс | Round Robin:   398.23 мс | Разница:    5.4%
Команд:  32 | Послед:   757.17 мс | Round Robin:   733.11 мс | Разница:   -3.2%



Отчёт сохранён: report_task19.txt
